# Module 1 — Data Understanding & Preparation

Seller-Level Vehicle Dataset (`seller_info_dm1_2627.csv`), one row per seller (`sp_id`).

Outputs of this notebook (used by Modules 2–4):
- `data/processed/clean_base.csv`: cleaned dataset; **missing values are kept as `NaN`** (no imputation here, to avoid leakage into Module 3).
- `data/processed/feature_sets.json`: retained/eliminated variables, log-transformed variables, and the reasons.
- `figures/m1_*.pdf|png`, `tables/m1_*.csv`: report material.

Conventions: `random_state=42`; units are **inferred** from value ranges (the dataset description does not state them).

In [1]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import plot_style as ps

ps.setup()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

RAW = ROOT / "data" / "raw" / "seller_info_dm1_2627.csv"
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
TARGET = "seller_rating_class"
SEED = 42

raw = pd.read_csv(RAW)
print(raw.shape)
raw.head()

(13843, 33)


,sp_id,n_listings,price_median,savings_amount_median,daysonmarket_median,mileage_median,year_median,vehicle_age_median,owner_count_median,horsepower_median,engine_displacement_median,city_fuel_economy_median,highway_fuel_economy_median,back_legroom_median,front_legroom_median,fuel_tank_volume_median,height_median,length_median,width_median,pct_isCab,pct_is_new,pct_salvage,pct_theft_title,n_listing_days,listing_span_days,dominant_body_type,dominant_fuel_type,dominant_transmission,dominant_wheel_system,dominant_listing_color,franchise_status,new_used_profile,seller_rating_class
0,404771.0,30,6700.0,431.0,13.0,193193.5,2011.0,9.0,2.0,258.5,NaN,17.0,23.5,38.65,41.3,21.0,68.9,195.35,77.10,0.133333,0.0,0.0,0.000000,15,156,suv_crossover,gasoline,a,NaN,gray,independent,used_only,medium
1,336590.0,17,10995.0,505.0,24.0,87589.0,2012.0,7.0,2.0,182.0,2400.0,21.0,28.5,37.30,41.4,15.9,64.7,180.30,72.50,0.058824,0.0,0.0,0.000000,14,589,suv_crossover,gasoline,a,awd,silver,independent,used_only,medium
2,392016.0,18,7645.0,632.0,43.0,151434.0,2010.5,9.5,2.0,205.0,NaN,20.0,28.5,35.50,41.7,17.5,58.1,188.20,72.00,0.111111,0.0,0.0,0.000000,15,97,sedan,gasoline,a,NaN,blue,independent,used_only,low
3,294197.0,12,23449.0,479.0,18.0,31186.0,2017.0,3.0,1.0,178.0,2500.0,25.0,35.0,38.65,42.3,16.5,57.9,190.05,72.05,0.000000,0.0,0.0,0.000000,7,338,sedan,gasoline,a,NaN,unknown,franchise,used_only,medium
4,372085.0,14,8990.0,505.5,213.0,144471.5,2007.5,12.0,1.5,240.0,3500.0,16.0,22.0,36.80,42.3,21.1,70.2,187.60,74.00,0.000000,0.0,0.0,0.071429,6,315,suv_crossover,gasoline,a,NaN,silver,independent,used_only,medium


## 1. Data semantics

Variables are grouped as in the dataset description. `sp_id` is an identifier (not a feature) and is stored as a float in the file; we cast it to an integer after checking that it has no fractional part.

In [2]:
assert (raw["sp_id"] % 1 == 0).all()
raw["sp_id"] = raw["sp_id"].astype("int64")

S = []  # (variable, group, type, unit, role)
def add(vars_, group, typ, unit, role="feature"):
    for v in vars_:
        S.append((v, group, typ, unit, role))

add(["sp_id"], "Identifier", "nominal (ID)", "-", "index only")
add(["n_listings"], "Listing activity", "discrete count", "listings")
add(["price_median"], "Vehicle stats (median)", "continuous", "USD")
add(["savings_amount_median"], "Vehicle stats (median)", "continuous", "USD")
add(["daysonmarket_median"], "Listing activity", "continuous (median of counts)", "days")
add(["mileage_median"], "Vehicle stats (median)", "continuous", "miles")
add(["year_median"], "Vehicle stats (median)", "discrete (interval)", "model year")
add(["vehicle_age_median"], "Vehicle stats (median)", "discrete", "years")
add(["owner_count_median"], "Vehicle stats (median)", "discrete", "owners")
add(["horsepower_median"], "Engine / economy (median)", "continuous", "hp")
add(["engine_displacement_median"], "Engine / economy (median)", "continuous", "cc")
add(["city_fuel_economy_median", "highway_fuel_economy_median"], "Engine / economy (median)", "continuous", "MPG (MPGe for EVs)")
add(["back_legroom_median", "front_legroom_median", "height_median", "length_median", "width_median"], "Dimensions (median)", "continuous", "inches")
add(["fuel_tank_volume_median"], "Dimensions (median)", "continuous", "gallons")
add(["pct_isCab", "pct_is_new", "pct_salvage", "pct_theft_title"], "Inventory composition", "bounded proportion", "[0, 1]")
add(["n_listing_days"], "Listing activity", "discrete count", "days")
add(["listing_span_days"], "Listing activity", "discrete count", "days")
add(["dominant_body_type", "dominant_fuel_type", "dominant_transmission", "dominant_wheel_system", "dominant_listing_color"], "Dominant categorical (mode)", "nominal", "-")
add(["franchise_status", "new_used_profile"], "Dealer profile", "nominal", "-")
add([TARGET], "Target", "ordinal (low < medium < high)", "-", "target")

sem = pd.DataFrame(S, columns=["variable", "group", "type", "unit", "role"])
assert set(sem.variable) == set(raw.columns) and len(sem) == raw.shape[1]
sem["dtype"] = sem.variable.map(raw.dtypes.astype(str))
sem["n_unique"] = sem.variable.map(raw.nunique())
sem.to_csv(ps.TAB_DIR / "m1_semantics.csv", index=False)
sem

,variable,group,type,unit,role,dtype,n_unique
0,sp_id,Identifier,nominal (ID),-,index only,int64,13843
1,n_listings,Listing activity,discrete count,listings,feature,int64,666
2,price_median,Vehicle stats (median),continuous,USD,feature,float64,7876
3,savings_amount_median,Vehicle stats (median),continuous,USD,feature,float64,2796
4,daysonmarket_median,Listing activity,continuous (median of counts),days,feature,float64,565
5,mileage_median,Vehicle stats (median),continuous,miles,feature,float64,9704
6,year_median,Vehicle stats (median),discrete (interval),model year,feature,float64,72
7,vehicle_age_median,Vehicle stats (median),discrete,years,feature,float64,71
8,owner_count_median,Vehicle stats (median),discrete,owners,feature,float64,10
9,horsepower_median,Engine / economy (median),continuous,hp,feature,float64,508


## 2. Distributions and statistics
### 2.1 Target class balance

In [3]:
cls = raw[TARGET].value_counts().reindex(ps.CLASS_ORDER)
cls_pct = (cls / cls.sum() * 100).round(1)
print(pd.DataFrame({"count": cls, "%": cls_pct}))
print("majority-class baseline accuracy:", round(cls.max() / cls.sum(), 3))

                     count     %
seller_rating_class             
low                   2579  18.6
medium                7577  54.7
high                  3687  26.6
majority-class baseline accuracy: 0.547


The target is **imbalanced** (medium ≈ 55%, high ≈ 27%, low ≈ 19%). A majority-class predictor already reaches ≈ 0.55 accuracy, so Module 3 must report macro-F1 and per-class recall, not only accuracy, and use stratified splits.

### 2.2 Numerical attributes

In [4]:
NUM = [c for c in raw.select_dtypes("number").columns if c != "sp_id"]
CAT = [c for c in raw.select_dtypes("object").columns if c != TARGET]

def describe_num(d, cols):
    t = pd.DataFrame({
        "count": d[cols].count(),
        "missing": d[cols].isna().sum(),
        "mean": d[cols].mean(),
        "std": d[cols].std(),
        "min": d[cols].min(),
        "q25": d[cols].quantile(.25),
        "median": d[cols].median(),
        "q75": d[cols].quantile(.75),
        "max": d[cols].max(),
        "skew": d[cols].skew(),
        "zeros": (d[cols] == 0).sum(),
    })
    t["IQR"] = t.q75 - t.q25
    return t

desc_raw = describe_num(raw, NUM)
desc_raw.round(3)

,count,missing,mean,std,min,q25,median,q75,max,skew,zeros,IQR
n_listings,13843,0,118.095,137.063,5.00,25.000,70.000,168.000,4959.000,5.247,0,143.000
price_median,13843,0,23106.140,14365.067,1699.00,12495.000,22385.000,29888.000,236438.000,3.446,0,17393.000
savings_amount_median,13843,0,446.213,519.462,0.00,0.000,358.000,734.000,9333.000,2.396,5100,734.000
daysonmarket_median,13843,0,49.147,56.060,0.00,26.000,36.000,51.000,1987.000,8.755,10,25.000
mileage_median,13842,1,51463.271,51845.462,0.00,25.000,33596.500,96915.750,238541.000,0.681,621,96890.750
year_median,13843,0,2015.949,4.722,1960.00,2013.000,2017.000,2020.000,2021.000,-2.253,0,7.000
vehicle_age_median,13842,1,4.083,4.633,0.00,0.000,3.000,7.000,60.000,2.322,4173,7.000
owner_count_median,13804,39,1.434,0.624,1.00,1.000,1.000,2.000,6.000,1.299,0,1.000
horsepower_median,13837,6,240.903,56.993,78.00,188.000,243.000,283.000,630.000,0.845,0,95.000
engine_displacement_median,10450,3393,2925.081,808.084,1200.00,2400.000,2500.000,3500.000,6800.000,1.294,0,1100.000


### 2.3 Categorical attributes

In [5]:
cat_tab = []
for c in CAT + [TARGET]:
    vc = raw[c].value_counts(dropna=False)
    cat_tab.append({"variable": c, "levels (incl. NaN)": vc.size,
                    "mode": vc.index[0], "mode %": round(vc.iloc[0] / len(raw) * 100, 1),
                    "NaN": int(raw[c].isna().sum()),
                    "'unknown'": int((raw[c] == "unknown").sum())})
    print(f"\n{c}\n{vc.to_string()}")
cat_tab = pd.DataFrame(cat_tab)
cat_tab.to_csv(ps.TAB_DIR / "m1_categorical_summary.csv", index=False)
cat_tab


dominant_body_type
dominant_body_type
suv_crossover    8920
sedan            3044
pickup_truck     1424
coupe             167
hatchback          97
van                82
convertible        63
minivan            39
wagon               7

dominant_fuel_type
dominant_fuel_type
gasoline             13658
flex_fuel_vehicle       91
diesel                  52
biodiesel               16
hybrid                  13
electric                12
unknown                  1

dominant_transmission
dominant_transmission
a          12976
cvt          847
m             14
unknown        6

dominant_wheel_system
dominant_wheel_system
NaN    10518
fwd     1781
awd      724
4wd      668
rwd      129
4x2       23

dominant_listing_color
dominant_listing_color
white      5167
black      4475
unknown    1510
silver      992
gray        875
red         417
blue        372
brown        17
green        12
gold          5
teal          1

franchise_status
franchise_status
franchise      6999
independent    6844



,variable,levels (incl. NaN),mode,mode %,NaN,'unknown'
0,dominant_body_type,9,suv_crossover,64.4,0,0
1,dominant_fuel_type,7,gasoline,98.7,0,1
2,dominant_transmission,4,a,93.7,0,6
3,dominant_wheel_system,6,NaN,76.0,10518,0
4,dominant_listing_color,11,white,37.3,0,1510
5,franchise_status,2,franchise,50.6,0,0
6,new_used_profile,3,used_only,53.3,0,0
7,seller_rating_class,3,medium,54.7,0,0


Observations:
- `franchise_status` has only **2** of the 4 levels listed in the description (`mixed`/`unknown` never occur); `new_used_profile` has no `unknown` level.
- `dominant_fuel_type` (98.7% gasoline) and `dominant_transmission` (93.7% automatic) are near-constant, so they carry little information.
- Several levels are very rare (e.g. `wagon` = 7 sellers, `teal` = 1); see the grouping in §4.

## 3. Data quality assessment
### 3.1 Duplicates

In [6]:
dup = {
    "exact duplicate rows": int(raw.duplicated().sum()),
    "duplicate sp_id": int(raw.sp_id.duplicated().sum()),
    "duplicate rows ignoring sp_id": int(raw.drop(columns="sp_id").duplicated().sum()),
}
dup

{'exact duplicate rows': 0,
 'duplicate sp_id': 0,
 'duplicate rows ignoring sp_id': 0}

### 3.2 Missing values (explicit `NaN` and implicit codes)

Implicit codes checked: the string `unknown`, empty strings, and impossible zeros. Zeros are **legitimate** in `savings_amount_median` (no discount), `mileage_median`/`vehicle_age_median` (new cars), `daysonmarket_median`, `listing_span_days` (single-day sellers), and the `pct_*` proportions, so they are not treated as missing. The zeros in `back_legroom_median` are discussed in §3.3.

In [7]:
miss = pd.DataFrame({
    "NaN": raw.isna().sum(),
    "unknown": (raw == "unknown").sum(),
    "empty string": (raw.astype(str).apply(lambda s: s.str.strip()) == "").sum(),
})
miss["total"] = miss.sum(axis=1)
miss["%"] = (miss.total / len(raw) * 100).round(2)
miss = miss[miss.total > 0].sort_values("total", ascending=False)
miss.to_csv(ps.TAB_DIR / "m1_missing.csv")
miss

,NaN,unknown,empty string,total,%
dominant_wheel_system,10518,0,0,10518,75.98
engine_displacement_median,3393,0,0,3393,24.51
dominant_listing_color,0,1510,0,1510,10.91
owner_count_median,39,0,0,39,0.28
pct_isCab,27,0,0,27,0.20
pct_theft_title,27,0,0,27,0.20
pct_salvage,27,0,0,27,0.20
back_legroom_median,19,0,0,19,0.14
city_fuel_economy_median,16,0,0,16,0.12
highway_fuel_economy_median,16,0,0,16,0.12


**Is missingness related to other variables?** For the variables with substantial missingness, we compare the missing rate across the target, franchise status, and inventory profile. This is descriptive only: no data-driven value is computed from it.

In [8]:
def miss_rate_by(mask, by):
    return mask.groupby(raw[by]).mean().round(3)

masks = {
    "dominant_wheel_system NaN": raw.dominant_wheel_system.isna(),
    "engine_displacement NaN": raw.engine_displacement_median.isna(),
    "listing_color = unknown": raw.dominant_listing_color.eq("unknown"),
}
mr = pd.concat({k: pd.concat({b: miss_rate_by(m, b) for b in [TARGET, "franchise_status", "new_used_profile"]})
                for k, m in masks.items()}, axis=1)
mr

dominant_wheel_system NaN  engine_displacement NaN  listing_color = unknown
seller_rating_class high                             0.761                    0.244                    0.102
                    low                              0.770                    0.258                    0.111
                    medium                           0.756                    0.241                    0.112
franchise_status    franchise                        0.755                    0.243                    0.159
                    independent                      0.765                    0.247                    0.058
new_used_profile    mixed                            0.758                    0.243                    0.159
                    new_only                         0.677                    0.226                    0.226
                    used_only                        0.761                    0.247                    0.065

In [9]:
# Chi-square test of independence: missing indicator vs. target
for k, m in masks.items():
    ct = pd.crosstab(m, raw[TARGET])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    print(f"{k:28s} vs target: chi2={chi2:7.2f}, dof={dof}, p={p:.3g}")
for k, m in masks.items():
    ct = pd.crosstab(m, raw["franchise_status"])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    print(f"{k:28s} vs franchise: chi2={chi2:7.2f}, dof={dof}, p={p:.3g}")

dominant_wheel_system NaN    vs target: chi2=   2.28, dof=2, p=0.32
engine_displacement NaN      vs target: chi2=   3.11, dof=2, p=0.211
listing_color = unknown      vs target: chi2=   2.41, dof=2, p=0.299
dominant_wheel_system NaN    vs franchise: chi2=   1.87, dof=1, p=0.171
engine_displacement NaN      vs franchise: chi2=   0.26, dof=1, p=0.608
listing_color = unknown      vs franchise: chi2= 362.30, dof=1, p=8.9e-81


**Joint missingness.** The 27 `NaN`s in `pct_isCab`, `pct_salvage` and `pct_theft_title` occur in exactly the same rows, and these are sellers whose inventory is (almost) entirely new. The source flags (cab / salvage / theft title) are undefined for new vehicles, so this missingness is structural.

In [10]:
pct_hist = ["pct_isCab", "pct_salvage", "pct_theft_title"]
m_pct = raw[pct_hist].isna()
print("same rows for all three:", (m_pct.nunique(axis=1) == 1).all())
print(raw.loc[m_pct.any(axis=1), "pct_is_new"].describe())
print(raw.loc[m_pct.any(axis=1), "new_used_profile"].value_counts())
print("\nowner_count NaN by profile:\n", raw.loc[raw.owner_count_median.isna(), "new_used_profile"].value_counts())
print("\nrows missing body dimensions:\n", raw.loc[raw.height_median.isna(), ["dominant_fuel_type", "dominant_body_type", "year_median"]])

same rows for all three: True


count    27.000000
mean      0.998051
std       0.010129
min       0.947368
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000
Name: pct_is_new, dtype: float64
new_used_profile
new_only    26
mixed        1
Name: count, dtype: int64

owner_count NaN by profile:
 new_used_profile
new_only     30
used_only     6
mixed         3
Name: count, dtype: int64

rows missing body dimensions:
       dominant_fuel_type dominant_body_type  year_median
2267            electric          hatchback       2017.0
3610             unknown              coupe       1969.0
9650            electric              sedan       2019.0
12432           electric          hatchback       2017.0
12545           electric              sedan       2017.0
13383           gasoline        convertible       1962.0


### 3.3 Syntactic and semantic consistency checks

Each rule is evaluated on the raw data. Violations are either **corrected** (clear data errors set to `NaN`), or **kept** when a domain explanation exists.

In [11]:
r = raw
is_ev = r.dominant_fuel_type.eq("electric")
implied_year = r.year_median + r.vehicle_age_median
maj_new = r.pct_is_new > 0.5

checks = [
    ("negative values in numeric columns", (r[NUM] < 0).any(axis=1), "none found"),
    ("pct_* outside [0, 1]", ((r[["pct_isCab", "pct_is_new", "pct_salvage", "pct_theft_title"]] < 0) | (r[["pct_isCab", "pct_is_new", "pct_salvage", "pct_theft_title"]] > 1)).any(axis=1), "none found"),
    ("new_used_profile inconsistent with pct_is_new", ~(((r.new_used_profile == "used_only") & (r.pct_is_new == 0)) | ((r.new_used_profile == "new_only") & (r.pct_is_new == 1)) | ((r.new_used_profile == "mixed") & r.pct_is_new.between(0, 1, inclusive="neither"))), "none found: profile is a deterministic function of pct_is_new"),
    ("n_listing_days > n_listings or > listing_span_days + 1", (r.n_listing_days > r.n_listings) | (r.n_listing_days > r.listing_span_days + 1), "none found"),
    ("savings_amount_median > price_median", r.savings_amount_median > r.price_median, "none found"),
    ("front_legroom_median > 46 in", r.front_legroom_median > 46, "ERROR -> NaN (passenger-car maximum is about 46 in; 54/67 in on an SUV is impossible)"),
    ("non-EV fuel economy > 60 MPG", ~is_ev & ((r.city_fuel_economy_median > 60) | (r.highway_fuel_economy_median > 60)), "ERROR -> NaN for city and highway (MPGe-like values on a gasoline V8/V6 seller)"),
    ("EV fuel economy > 60 (MPGe)", is_ev & (r.city_fuel_economy_median > 60), "kept: MPGe is a valid EV measure"),
    ("highway < city fuel economy", r.highway_fuel_economy_median < r.city_fuel_economy_median, "kept: normal for hybrids/EVs and mixed inventories"),
    ("back_legroom_median == 0", r.back_legroom_median == 0, "kept: regular-cab pickups / cargo vans have no rear seat"),
    ("year_median < 1995 (classic cars)", r.year_median < 1995, "kept: real classic-car dealers (coupes/convertibles, high prices)"),
    ("year + age (median) outside 2018-2022", ~implied_year.between(2018, 2022), "kept: medians are not additive, so this is only a soft check"),
    ("pct_is_new > 0.5 but mileage_median > 5000", maj_new & (r.mileage_median > 5000), "kept: is_new flags recent model years (age 0), not zero mileage"),
    ("daysonmarket_median > listing_span_days", r.daysonmarket_median > r.listing_span_days, "kept: days on market can start before the observation window"),
]
chk = pd.DataFrame([(n, int(m.sum()), a) for n, m, a in checks], columns=["rule", "violations", "action"])
chk.to_csv(ps.TAB_DIR / "m1_consistency_checks.csv", index=False)
chk

,rule,violations,action
0,negative values in numeric columns,0,none found
1,"pct_* outside [0, 1]",0,none found
2,new_used_profile inconsistent with pct_is_new,0,none found: profile is a deterministic functio...
3,n_listing_days > n_listings or > listing_span_...,0,none found
4,savings_amount_median > price_median,0,none found
5,front_legroom_median > 46 in,2,ERROR -> NaN (passenger-car maximum is about 4...
6,non-EV fuel economy > 60 MPG,2,ERROR -> NaN for city and highway (MPGe-like v...
7,EV fuel economy > 60 (MPGe),8,kept: MPGe is a valid EV measure
8,highway < city fuel economy,18,kept: normal for hybrids/EVs and mixed invento...
9,back_legroom_median == 0,3,kept: regular-cab pickups / cargo vans have no...


In [12]:
# Evidence for the "kept" decisions
print("Majority-new sellers with mileage > 5000: vehicle_age_median distribution")
print(r.loc[maj_new & (r.mileage_median > 5000), "vehicle_age_median"].describe().round(2))
print("\nback_legroom == 0:\n", r.loc[r.back_legroom_median == 0, ["dominant_body_type", "length_median", "front_legroom_median"]])
print("\nfront_legroom > 46:\n", r.loc[r.front_legroom_median > 46, ["dominant_body_type", "front_legroom_median", "length_median", "height_median"]])
print("\nnon-EV fuel economy > 60:\n", r.loc[~is_ev & (r.city_fuel_economy_median > 60), ["dominant_fuel_type", "dominant_body_type", "horsepower_median", "engine_displacement_median", "city_fuel_economy_median", "highway_fuel_economy_median"]])
print("\nclassic sellers (year < 1995): body types\n", r.loc[r.year_median < 1995, "dominant_body_type"].value_counts())
print("median price classic vs rest:", r.loc[r.year_median < 1995, "price_median"].median(), r.loc[r.year_median >= 1995, "price_median"].median())

Majority-new sellers with mileage > 5000: vehicle_age_median distribution
count    338.00
mean       0.22
std        0.51
min        0.00
25%        0.00
50%        0.00
75%        0.00
max        3.00
Name: vehicle_age_median, dtype: float64

back_legroom == 0:
       dominant_body_type  length_median  front_legroom_median
1699        pickup_truck         227.00                  41.3
1954        pickup_truck         226.75                  41.1
10504                van         219.90                  41.3

front_legroom > 46:
       dominant_body_type  front_legroom_median  length_median  height_median
10159      suv_crossover                 54.35          184.9           63.9
10396      suv_crossover                 67.00          191.1           63.9

non-EV fuel economy > 60:
      dominant_fuel_type dominant_body_type  horsepower_median  engine_displacement_median  city_fuel_economy_median  highway_fuel_economy_median
557            gasoline       pickup_truck              387.5 

## 4. Cleaning

Every action is **row-local or rule-based**: no statistic is estimated from the data, so the cleaned base can safely be split later without leakage.

| # | Action | Rows affected | Justification |
|---|---|---|---|
| C1 | Drop `dominant_wheel_system` | all | 76% missing; the rate is unrelated to target/franchise (MCAR-like), so imputing it would invent most of the column |
| C2 | `front_legroom_median > 46` → `NaN` | 2 | physically impossible |
| C3 | non-EV fuel economy > 60 → `NaN` (city and highway) | 2 | impossible for gasoline engines |
| C4 | `pct_isCab`, `pct_salvage`, `pct_theft_title` `NaN` → 0 | 27 | structural: new vehicles cannot be ex-cabs, salvage or theft titles (deterministic domain rule) |
| C5 | `unknown` in fuel/transmission → `NaN` | 7 | true missing values, too rare to be a category |
| C6 | Rare levels grouped (domain-based) | see below | levels with < 1% of sellers make unstable splits, one-hot columns and itemsets |

`dominant_listing_color = unknown` (10.9%) is **kept as its own level**: its rate differs strongly between franchise (15.9%) and independent (5.8%) sellers, so it is informative (missing at random given the dealer type), not noise.

All other `NaN`s (`engine_displacement_median` 24.5%, `owner_count_median` 0.3%, a handful of dimension/engine values for EV and classic sellers) are **left as `NaN`**. They are imputed later, fit on the full data for the unsupervised modules (M2/M4) and inside the training folds for M3.

In [13]:
df = raw.copy().set_index("sp_id")

df = df.drop(columns="dominant_wheel_system")                                    # C1
df.loc[df.front_legroom_median > 46, "front_legroom_median"] = np.nan            # C2
bad_fe = (df.dominant_fuel_type != "electric") & ((df.city_fuel_economy_median > 60) | (df.highway_fuel_economy_median > 60))
df.loc[bad_fe, ["city_fuel_economy_median", "highway_fuel_economy_median"]] = np.nan  # C3
pct_nan = df[pct_hist].isna().all(axis=1)
assert (df.loc[pct_nan, "pct_is_new"] >= 0.9).all()  # rule only applied to (almost) all-new sellers
df.loc[pct_nan, pct_hist] = 0.0                                                   # C4
for c in ["dominant_fuel_type", "dominant_transmission"]:                         # C5
    df[c] = df[c].replace("unknown", np.nan)

GROUPS = {                                                                         # C6
    "dominant_body_type": {"coupe": "coupe_convertible", "convertible": "coupe_convertible",
                           "van": "van_minivan", "minivan": "van_minivan",
                           "hatchback": "hatchback_wagon", "wagon": "hatchback_wagon"},
    "dominant_fuel_type": {"biodiesel": "diesel", "hybrid": "electrified", "electric": "electrified"},
    "dominant_listing_color": {"brown": "other", "green": "other", "gold": "other", "teal": "other"},
}
for c, m in GROUPS.items():
    df[c] = df[c].replace(m)

print(df.shape)
for c in ["dominant_body_type", "dominant_fuel_type", "dominant_transmission", "dominant_listing_color"]:
    print(f"\n{df[c].value_counts(dropna=False).to_string()}")

(13843, 31)

dominant_body_type
suv_crossover        8920
sedan                3044
pickup_truck         1424
coupe_convertible     230
van_minivan           121
hatchback_wagon       104

dominant_fuel_type
gasoline             13658
flex_fuel_vehicle       91
diesel                  68
electrified             25
NaN                      1

dominant_transmission
a      12976
cvt      847
m         14
NaN        6

dominant_listing_color
white      5167
black      4475
unknown    1510
silver      992
gray        875
red         417
blue        372
other        35


`dominant_transmission = m` (14 sellers, 0.1%) is kept as a separate level: a manual gearbox is semantically distinct and cannot be merged with automatic or CVT. Downstream modules handle it through `min_sup` (M4) or one-hot encoding (M3).

### 4.1 Outliers

Univariate outliers are counted with the IQR rule (1.5·IQR), on the log scale for the variables selected for a log transform in the next cell (the raw scale flags the whole long tail of skewed counts). They are **not removed**: the inspections above show that the extremes are real seller types (supercar dealers up to $236k, classic-car dealers, a 4,959-listing high-volume dealer, heavy-duty trucks/high-roof vans, EV sellers with MPGe). The corrupted values were already fixed in §4. Multivariate anomalies are studied in Module 2 through DBSCAN noise points. Robust choices such as log scaling or robust scalers are made per module.

In [14]:
LOG_CANDIDATES = ["n_listings", "price_median", "savings_amount_median", "daysonmarket_median",
                  "mileage_median", "listing_span_days", "n_listing_days", "horsepower_median",
                  "engine_displacement_median", "city_fuel_economy_median", "highway_fuel_economy_median",
                  "fuel_tank_volume_median", "length_median", "vehicle_age_median", "owner_count_median"]
# proportions ([0,1]), year (interval scale) and dimensions with no right tail are not candidates

def skew_after_log(s):
    return np.log1p(s.dropna()).skew()

tr = pd.DataFrame({"skew_raw": df[LOG_CANDIDATES].skew(),
                   "skew_log1p": [skew_after_log(df[c]) for c in LOG_CANDIDATES]})
# Rule: log1p if the raw distribution is clearly right-skewed (skew > 1)
# AND the log halves the absolute skew (otherwise the transform only costs interpretability)
tr["log1p"] = (tr.skew_raw > 1) & (tr.skew_log1p.abs() < 0.5 * tr.skew_raw.abs())
tr.to_csv(ps.TAB_DIR / "m1_log_transform.csv")
LOG_VARS = tr.index[tr.log1p].tolist()
print("log1p selected:", LOG_VARS)
tr.round(2)

log1p selected: ['n_listings', 'price_median', 'savings_amount_median', 'daysonmarket_median', 'engine_displacement_median', 'city_fuel_economy_median', 'highway_fuel_economy_median', 'fuel_tank_volume_median', 'vehicle_age_median']


,skew_raw,skew_log1p,log1p
n_listings,5.25,-0.13,True
price_median,3.45,-0.45,True
savings_amount_median,2.40,-0.40,True
daysonmarket_median,8.75,0.47,True
mileage_median,0.68,-0.91,False
listing_span_days,2.15,-1.22,False
n_listing_days,0.98,-0.47,False
horsepower_median,0.85,0.14,False
engine_displacement_median,1.29,0.40,True
city_fuel_economy_median,8.08,0.90,True


In [15]:
NUM_CLEAN = [c for c in df.select_dtypes("number").columns]
def iqr_outliers(s):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return int(((s < q1 - 1.5 * i) | (s > q3 + 1.5 * i)).sum())

out = pd.DataFrame({
    "scale": ["log1p" if c in LOG_VARS else "raw" for c in NUM_CLEAN],
    "IQR outliers (raw scale)": [iqr_outliers(df[c].dropna()) for c in NUM_CLEAN],
    "IQR outliers (used scale)": [iqr_outliers(np.log1p(df[c].dropna()) if c in LOG_VARS else df[c].dropna()) for c in NUM_CLEAN],
}, index=NUM_CLEAN)
out["% (used scale)"] = (out["IQR outliers (used scale)"] / df[NUM_CLEAN].notna().sum() * 100).round(2)
out.to_csv(ps.TAB_DIR / "m1_outliers.csv")
out

,scale,IQR outliers (raw scale),IQR outliers (used scale),% (used scale)
n_listings,log1p,641,1,0.01
price_median,log1p,171,70,0.51
savings_amount_median,log1p,218,0,0.00
daysonmarket_median,log1p,1110,842,6.08
mileage_median,raw,0,0,0.00
year_median,raw,65,65,0.47
vehicle_age_median,log1p,62,0,0.00
owner_count_median,raw,51,51,0.37
horsepower_median,raw,62,62,0.45
engine_displacement_median,log1p,314,60,0.57


Notes on the IQR counts: for zero-inflated proportions (`pct_salvage` 91% zeros, `pct_theft_title` 85% zeros) IQR = 0, so *every* non-zero value is flagged. The IQR rule is not meaningful for them, and Modules 2–3 should consider binary indicators (`any salvage`) instead. `pct_is_new` is bimodal (53% exact zeros, then a spread of mixed dealers), which is a structural trait and not an outlier pattern.

Limits of the log transform: `savings_amount_median` has 37% exact zeros (all `new_only` sellers and 74% of `mixed` ones, versus 5% of `used_only`), so log1p compresses the tail but the spike at 0 remains. The log scale still leaves 6% IQR outliers in `daysonmarket_median`, from the long-unsold inventories (> 1 year). Both are real seller behaviour and are kept.

## 5. Distribution plots

In [16]:
# Figure 1: class balance and target vs. franchise status
fig, axes = plt.subplots(1, 2, figsize=(ps.FULL_W, 2.2), gridspec_kw={"width_ratios": [1, 1.4]})
ax = axes[0]
ax.bar(ps.CLASS_ORDER, cls.values, color=[ps.CLASS_COLORS[c] for c in ps.CLASS_ORDER], width=0.6)
for i, (n, p) in enumerate(zip(cls.values, cls_pct.values)):
    ax.text(i, n + 120, f"{n:,}\n({p}%)", ha="center", va="bottom", fontsize=7, color=ps.INK)
ax.set_ylim(0, cls.max() * 1.3)
ax.set_ylabel("sellers"); ax.set_title("seller_rating_class distribution"); ax.grid(axis="x", visible=False)

ax = axes[1]
ct = pd.crosstab(df.franchise_status, df[TARGET], normalize="index")[ps.CLASS_ORDER]
left = np.zeros(len(ct))
for c in ps.CLASS_ORDER:
    ax.barh(ct.index, ct[c], left=left, color=ps.CLASS_COLORS[c], label=c, height=0.55, edgecolor="white", linewidth=2)
    for y, (l, w) in enumerate(zip(left, ct[c])):
        ax.text(l + w / 2, y, f"{w:.0%}", ha="center", va="center", fontsize=7,
                color="white" if c != "low" else ps.INK)
    left += ct[c].values
ax.set_xlim(0, 1); ax.set_xlabel("share of sellers"); ax.set_title("rating class by franchise_status")
ax.legend(ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.12), frameon=False)
ax.grid(axis="y", visible=False)
fig.tight_layout()
ps.save(fig, "m1_target_balance")
print(pd.crosstab(df.franchise_status, df[TARGET], normalize="index").round(3))
print(stats.chi2_contingency(pd.crosstab(df.franchise_status, df[TARGET]))[:2])

seller_rating_class   high    low  medium
franchise_status                         
franchise            0.206  0.209   0.585
independent          0.328  0.163   0.509
(270.33940992865263, 1.9794430600230783e-59)


In [17]:
# Figure 2: missing values per variable (after cleaning C1-C5; dominant_wheel_system shown from raw)
m_after = pd.DataFrame({"NaN": df.isna().sum(), "unknown (kept as level)": (df == "unknown").sum()})
m_after.loc["dominant_wheel_system (dropped)", "NaN"] = raw.dominant_wheel_system.isna().sum()
m_after.loc["dominant_wheel_system (dropped)", "unknown (kept as level)"] = 0
m_after = m_after[(m_after.sum(axis=1)) > 0]
m_after = (m_after / len(df) * 100)
m_after["total"] = m_after.sum(axis=1)
m_after = m_after.sort_values("total")
is_unk = m_after["unknown (kept as level)"] > 0
# each variable has only one kind of missingness, so one bar per row, colored by kind
labels = [i.replace("_median", "") for i in m_after.index]
fig, ax = plt.subplots(figsize=(ps.COL_W, 2.8))
ax.barh(labels, m_after.total, left=0.005, height=0.6,
        color=[ps.CATEGORICAL[1] if u else ps.CATEGORICAL[0] for u in is_unk])
for y, v in enumerate(m_after.total):
    ax.text(v * 1.25, y, f"{v:.2f}%" if v < 1 else f"{v:.1f}%", va="center", fontsize=6, color=ps.INK_2)
ax.set_xscale("log"); ax.set_xlim(0.005, 400)
ax.set_xticks([0.01, 0.1, 1, 10, 100]); ax.set_xticklabels(["0.01", "0.1", "1", "10", "100"])
ax.set_xlabel("% of sellers (log scale)")  # title goes in the LaTeX caption
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=ps.CATEGORICAL[0], label="NaN"), Patch(color=ps.CATEGORICAL[1], label="'unknown' (kept as level)")],
          loc="lower center", bbox_to_anchor=(0.35, 1.04), ncol=2, frameon=False)
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", labelsize=6.5)
fig.tight_layout()
ps.save(fig, "m1_missing")
m_after.round(2)

,NaN,unknown (kept as level),total
mileage_median,0.01,0.00,0.01
vehicle_age_median,0.01,0.00,0.01
dominant_fuel_type,0.01,0.00,0.01
dominant_transmission,0.04,0.00,0.04
horsepower_median,0.04,0.00,0.04
width_median,0.04,0.00,0.04
length_median,0.04,0.00,0.04
height_median,0.04,0.00,0.04
fuel_tank_volume_median,0.05,0.00,0.05
front_legroom_median,0.07,0.00,0.07


In [18]:
# Figure 3: raw vs log1p distributions of the log-transformed variables that are modelling features
# (year/age/engine/highway are excluded from modelling in §6, so they are not plotted)
PLOT_LOG = [c for c in LOG_VARS if c not in ["vehicle_age_median", "engine_displacement_median", "highway_fuel_economy_median"]]
TITLES = {"n_listings": "n_listings", "price_median": "price", "savings_amount_median": "savings",
          "daysonmarket_median": "days on market", "city_fuel_economy_median": "city fuel econ.",
          "fuel_tank_volume_median": "fuel tank"}
fig, axes = plt.subplots(2, len(PLOT_LOG), figsize=(ps.FULL_W, 3.0))
for j, c in enumerate(PLOT_LOG):
    s = df[c].dropna()
    hi = s.quantile(0.995)  # raw panel clipped at p99.5 so the bulk is visible; skew is computed on all values
    axes[0, j].hist(s[s <= hi], bins=40, color=ps.CATEGORICAL[0])
    axes[0, j].set_title(TITLES[c] + (" ($k)" if c == "price_median" else ""), fontsize=7.5)
    if c == "price_median":
        axes[0, j].xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v / 1000:.0f}"))
    axes[1, j].hist(np.log1p(s), bins=40, color=ps.CATEGORICAL[1])
    for a, v in [(axes[0, j], s.skew()), (axes[1, j], np.log1p(s).skew())]:
        a.text(0.97, 0.95, f"skew {v:.1f}", transform=a.transAxes, ha="right", va="top", fontsize=6.5, color=ps.INK_2,
               bbox=dict(facecolor="white", edgecolor="none", pad=1))
        a.tick_params(labelsize=6); a.grid(axis="x", visible=False)
        a.xaxis.set_major_locator(plt.MaxNLocator(3))
        a.yaxis.set_major_locator(plt.MaxNLocator(4))
axes[0, 0].set_ylabel("raw (≤ p99.5)\nsellers"); axes[1, 0].set_ylabel("log1p\nsellers")
fig.tight_layout()
ps.save(fig, "m1_log_transform")

In [19]:
# Figure 4: key variables by rating class (log axis where the variable is log-transformed)
KEY = ["n_listings", "price_median", "mileage_median", "vehicle_age_median",
       "pct_is_new", "savings_amount_median", "daysonmarket_median", "horsepower_median"]
fig, axes = plt.subplots(2, 4, figsize=(ps.FULL_W, 3.6))
for ax, c in zip(axes.ravel(), KEY):
    data = [df.loc[df[TARGET] == k, c].dropna() for k in ps.CLASS_ORDER]
    if c in LOG_VARS:
        data = [np.log1p(d) for d in data]
    bp = ax.boxplot(data, widths=0.55, patch_artist=True, showfliers=True,
                    flierprops=dict(marker="o", markersize=1.5, alpha=0.3, markeredgecolor="none", markerfacecolor=ps.NEUTRAL),
                    medianprops=dict(color="white", linewidth=1.5),
                    whiskerprops=dict(color=ps.INK_2, linewidth=0.8), capprops=dict(color=ps.INK_2, linewidth=0.8))
    for patch, k in zip(bp["boxes"], ps.CLASS_ORDER):
        patch.set_facecolor(ps.CLASS_COLORS[k]); patch.set_edgecolor("none")
    bp["medians"][0].set_color(ps.INK)  # light box -> dark median line
    ax.set_xticks([1, 2, 3]); ax.set_xticklabels(ps.CLASS_ORDER)
    ax.set_title(("log1p " if c in LOG_VARS else "") + c.replace("_median", "").replace("_", " "), fontsize=7.5)
    ax.grid(axis="x", visible=False)
fig.tight_layout()
ps.save(fig, "m1_box_by_target")

# Kruskal-Wallis (non-parametric, skewed data) + effect size epsilon^2 for every numeric variable
kw = []
for c in NUM_CLEAN:
    g = [df.loc[df[TARGET] == k, c].dropna() for k in ps.CLASS_ORDER]
    H, p = stats.kruskal(*g)
    N = sum(len(x) for x in g)
    kw.append((c, H, p, H / (N - 1), *[x.median() for x in g]))
kw = pd.DataFrame(kw, columns=["variable", "H", "p", "epsilon2", "median_low", "median_medium", "median_high"]).set_index("variable").sort_values("epsilon2", ascending=False)
kw.to_csv(ps.TAB_DIR / "m1_kruskal_target.csv")
kw.round(4)

,H,p,epsilon2,median_low,median_medium,median_high
variable,,,,,,
n_listings,1133.4884,0.0000,0.0819,107.0000,88.0000,34.0000
n_listing_days,1012.3080,0.0000,0.0731,49.0000,45.0000,24.0000
year_median,228.8600,0.0000,0.0165,2018.0000,2018.0000,2016.0000
pct_is_new,221.3757,0.0000,0.0160,0.2164,0.0000,0.0000
vehicle_age_median,213.2718,0.0000,0.0154,2.0000,2.0000,4.0000
mileage_median,169.8895,0.0000,0.0123,29038.0000,30845.0000,48603.0000
owner_count_median,163.6646,0.0000,0.0119,1.0000,1.0000,1.0000
listing_span_days,142.6485,0.0000,0.0103,384.0000,367.0000,329.0000
city_fuel_economy_median,124.6202,0.0000,0.0090,21.0000,20.0000,20.0000


All differences are statistically significant except `front_legroom` (n ≈ 13.8k makes p-values uninformative), but the effect sizes are weak: ε² ≈ 0.08 for seller size and ≤ 0.017 for every other variable. The strongest separation comes from seller size and activity (`n_listings`, `n_listing_days`) and from inventory age/newness (`vehicle_age`, `year`, `mileage`, `pct_is_new`). **High-rated sellers tend to be small, used-car, mostly independent dealers**, while low and medium sellers have very similar distributions. This already explains why `seller_rating_class` is a "hard" task, with `low` vs `medium` the most difficult distinction.

In [20]:
# Figure 5: categorical distributions (after grouping)
CATS = ["dominant_body_type", "dominant_listing_color", "dominant_transmission", "new_used_profile"]
fig, axes = plt.subplots(1, 4, figsize=(ps.FULL_W, 2.0))
for ax, c in zip(axes, CATS):
    vc = df[c].value_counts(dropna=False).sort_values()
    labels = ["NaN" if pd.isna(i) else str(i) for i in vc.index]
    pct = vc.values / len(df) * 100
    ax.barh(labels, pct, color=ps.CATEGORICAL[0], height=0.6)
    for y, (p, k) in enumerate(zip(pct, vc.values)):
        ax.text(p + 1.5, y, f"{k:,}", va="center", fontsize=5.5, color=ps.INK_2)
    ax.set_xlim(0, pct.max() * 1.35)
    ax.set_title(c.replace("dominant_", ""), fontsize=7.5); ax.set_xlabel("% sellers (labels: counts)", fontsize=6.5)
    ax.tick_params(axis="y", labelsize=6.5); ax.grid(axis="y", visible=False)
fig.tight_layout()
ps.save(fig, "m1_categorical")

## 6. Pairwise correlations and variable elimination

In [21]:
num_c = [c for c in NUM_CLEAN]
spear = df[num_c].corr("spearman")
pear = df[num_c].corr("pearson")

# Figure 6: Spearman heatmap (lower triangle)
short = {c: c.replace("_median", "").replace("_fuel_economy", "_fe").replace("listing_", "lst_").replace("amount", "amt") for c in num_c}
mask = np.triu(np.ones_like(spear, dtype=bool))
fig, ax = plt.subplots(figsize=(ps.FULL_W, 5.6))
im = ax.imshow(np.ma.masked_where(mask, spear.values), cmap=ps.DIVERGING, vmin=-1, vmax=1)
ax.set_xticks(range(len(num_c))); ax.set_xticklabels([short[c] for c in num_c], rotation=60, ha="right", fontsize=6.5)
ax.set_yticks(range(len(num_c))); ax.set_yticklabels([short[c] for c in num_c], fontsize=6.5)
for i in range(len(num_c)):
    for j in range(i):
        v = spear.values[i, j]
        if abs(v) >= 0.5:
            ax.text(j, i, f"{v:.2f}".replace("0.", "."), ha="center", va="center", fontsize=5,
                    color="white" if abs(v) >= 0.75 else ps.INK)
ax.grid(False)
for s in ax.spines.values(): s.set_visible(False)
cb = fig.colorbar(im, ax=ax, shrink=0.6, pad=0.02); cb.set_label("Spearman ρ"); cb.outline.set_visible(False)
ax.set_title("Spearman correlation (|ρ| ≥ 0.5 annotated)")
fig.tight_layout()
ps.save(fig, "m1_spearman_heatmap")

In [22]:
pairs = []
for i, a in enumerate(num_c):
    for b in num_c[i + 1:]:
        pairs.append((a, b, pear.loc[a, b], spear.loc[a, b]))
pairs = pd.DataFrame(pairs, columns=["var_a", "var_b", "pearson", "spearman"])
pairs["max_abs"] = pairs[["pearson", "spearman"]].abs().max(axis=1)
pairs = pairs.sort_values("max_abs", ascending=False)
top_pairs = pairs[pairs.max_abs >= 0.8]
top_pairs.to_csv(ps.TAB_DIR / "m1_top_correlations.csv", index=False)
top_pairs.round(3)

,var_a,var_b,pearson,spearman,max_abs
105,year_median,vehicle_age_median,-0.996,-0.984,0.996
21,n_listings,n_listing_days,0.839,0.956,0.956
185,city_fuel_economy_median,highway_fuel_economy_median,0.936,0.936,0.936
86,mileage_median,year_median,-0.826,-0.932,0.932
87,mileage_median,vehicle_age_median,0.820,0.927,0.927
118,year_median,pct_is_new,0.684,0.873,0.873
135,vehicle_age_median,pct_is_new,-0.678,-0.867,0.867
100,mileage_median,pct_is_new,-0.757,-0.861,0.861
232,fuel_tank_volume_median,length_median,0.841,0.760,0.841
175,engine_displacement_median,fuel_tank_volume_median,0.826,0.810,0.826


Spearman is used as the primary criterion because most variables are skewed or zero-inflated and the relations are monotonic but not linear (e.g. price vs mileage: Pearson −0.68, Spearman −0.82). We also report Pearson for comparison.

**Elimination rule:** a pair with |ρ_s| ≥ 0.90 is considered redundant, and one member is dropped from the **modelling feature set**. All variables stay in `clean_base.csv` for profiling and for the regression task.

| Pair (ρ_s) | Kept | Dropped | Reason |
|---|---|---|---|
| year ↔ vehicle_age (−0.98) | – | both (see next row) | age = listing year − model year: the same information |
| mileage ↔ year (−0.93), mileage ↔ age (+0.93) | `mileage_median` | `year_median`, `vehicle_age_median` | mileage is the most direct wear measure, is continuous, and has the lowest skew of the three (0.68 vs 2.3); age is kept for profiling |
| n_listings ↔ n_listing_days (+0.96) | `n_listings` | `n_listing_days` | both measure seller size/activity; n_listings is the more direct volume measure |
| city_fe ↔ highway_fe (+0.94) | `city_fuel_economy_median` | `highway_fuel_economy_median` | near-identical information |

Additional eliminations, not driven by the threshold:
- `engine_displacement_median`: 24.5% missing, and its information is already covered by `fuel_tank_volume_median` (ρ_s = 0.81), `city_fuel_economy_median` (−0.79) and `horsepower_median`. Imputing a quarter of the column would add noise and no new information.
- `new_used_profile`: a deterministic discretization of `pct_is_new` (verified in §3.3), so it is redundant with it; it is kept for profiling and for M4 items.

Pairs with 0.85 ≤ |ρ_s| < 0.90 that are **kept**: `mileage` ↔ `pct_is_new` (−0.86; Pearson only −0.76). Both are driven by the new-car mass at zero mileage, but they describe different things (inventory composition vs vehicle wear).

In [23]:
# Categorical associations (Cramér's V, bias-corrected)
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum(); r, k = ct.shape
    phi2 = max(0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    rc, kc = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    return np.sqrt(phi2 / max(1e-12, min(kc - 1, rc - 1)))

CAT_CLEAN = ["dominant_body_type", "dominant_fuel_type", "dominant_transmission", "dominant_listing_color",
             "franchise_status", "new_used_profile", TARGET]
cv = pd.DataFrame(index=CAT_CLEAN, columns=CAT_CLEAN, dtype=float)
for a in CAT_CLEAN:
    for b in CAT_CLEAN:
        if a == b:
            cv.loc[a, b] = 1.0
            continue
        sub = df[[a, b]].dropna()
        cv.loc[a, b] = cramers_v(sub[a], sub[b])
cv.to_csv(ps.TAB_DIR / "m1_cramers_v.csv")
print(pd.crosstab(df.franchise_status, df.new_used_profile))
cv.round(3)

new_used_profile  mixed  new_only  used_only
franchise_status                            
franchise          6358        31        610
independent          70         0       6774


,dominant_body_type,dominant_fuel_type,dominant_transmission,dominant_listing_color,franchise_status,new_used_profile,seller_rating_class
dominant_body_type,1.000,0.216,0.162,0.121,0.379,0.255,0.069
dominant_fuel_type,0.216,1.000,0.030,0.049,0.107,0.070,0.026
dominant_transmission,0.162,0.030,1.000,0.129,0.232,0.160,0.054
dominant_listing_color,0.121,0.049,0.129,1.000,0.297,0.211,0.065
franchise_status,0.379,0.107,0.232,0.297,1.000,0.905,0.139
new_used_profile,0.255,0.070,0.160,0.211,0.905,1.000,0.090
seller_rating_class,0.069,0.026,0.054,0.065,0.139,0.090,1.000


`franchise_status` and `new_used_profile` are strongly associated (Cramér's V shown above): 91% of franchise dealers have a mixed new/used inventory and 99% of independents are used-only. Both are kept because they are only two columns and the 680 sellers that break the pattern (franchise and used-only, or independent and mixed) can be informative. Since `new_used_profile` is already excluded from the numeric/modelling set (it is redundant with `pct_is_new`), only `franchise_status` enters the modelling features.

## 7. Save the clean base and the feature-set definitions

In [24]:
DROP_REDUNDANT = ["year_median", "vehicle_age_median", "n_listing_days", "highway_fuel_economy_median",
                  "engine_displacement_median"]
NUM_MODEL = [c for c in NUM_CLEAN if c not in DROP_REDUNDANT]
CAT_MODEL = ["dominant_body_type", "dominant_fuel_type", "dominant_transmission", "dominant_listing_color",
             "franchise_status"]

# re-check: no retained pair is above the threshold
sp_model = df[NUM_MODEL].corr("spearman").abs().where(~np.eye(len(NUM_MODEL), dtype=bool))
assert sp_model.max().max() < 0.90, sp_model.max().max()
print("max |rho_s| among retained numeric features:", round(sp_model.max().max(), 3))

feature_sets = {
    "target": TARGET,
    "id": "sp_id",
    "numeric_model": NUM_MODEL,
    "categorical_model": CAT_MODEL,
    "log1p_vars": LOG_VARS,
    "dropped_from_base": {"dominant_wheel_system": "76% missing, MCAR-like"},
    "excluded_from_model": {
        "year_median": "|rho_s|=0.98 with vehicle_age, 0.93 with mileage",
        "vehicle_age_median": "|rho_s|=0.93 with mileage (kept for profiling)",
        "n_listing_days": "|rho_s|=0.96 with n_listings",
        "highway_fuel_economy_median": "|rho_s|=0.94 with city fuel economy",
        "engine_displacement_median": "24.5% missing + redundant with fuel tank/fuel economy/horsepower",
        "new_used_profile": "deterministic discretization of pct_is_new",
    },
    "category_groupings": GROUPS,
    "notes": "clean_base keeps NaN; impute in downstream pipelines (M3: inside CV folds).",
}
with open(OUT / "feature_sets.json", "w") as f:
    json.dump(feature_sets, f, indent=2)

df.to_csv(OUT / "clean_base.csv")
print(df.shape, "->", OUT / "clean_base.csv")
print("numeric model features:", len(NUM_MODEL), NUM_MODEL)
print("categorical model features:", CAT_MODEL)
print("remaining NaN in model features:\n", df[NUM_MODEL + CAT_MODEL].isna().sum()[lambda s: s > 0])

max |rho_s| among retained numeric features: 0.861
(13843, 31) -> D:\Data Mining\data\processed\clean_base.csv
numeric model features: 19 ['n_listings', 'price_median', 'savings_amount_median', 'daysonmarket_median', 'mileage_median', 'owner_count_median', 'horsepower_median', 'city_fuel_economy_median', 'back_legroom_median', 'front_legroom_median', 'fuel_tank_volume_median', 'height_median', 'length_median', 'width_median', 'pct_isCab', 'pct_is_new', 'pct_salvage', 'pct_theft_title', 'listing_span_days']
categorical model features: ['dominant_body_type', 'dominant_fuel_type', 'dominant_transmission', 'dominant_listing_color', 'franchise_status']
remaining NaN in model features:
 mileage_median               1
owner_count_median          39
horsepower_median            6
city_fuel_economy_median    18
back_legroom_median         19
front_legroom_median         9
fuel_tank_volume_median      7
height_median                6
length_median                6
width_median                 6


In [25]:
# Final descriptive table on the clean base (for the report)
desc_clean = describe_num(df, NUM_CLEAN)
desc_clean.round(3).to_csv(ps.TAB_DIR / "m1_numeric_summary.csv")
desc_clean[["count", "missing", "mean", "std", "min", "median", "IQR", "max", "skew"]].round(2)

,count,missing,mean,std,min,median,IQR,max,skew
n_listings,13843,0,118.10,137.06,5.00,70.00,143.00,4959.00,5.25
price_median,13843,0,23106.14,14365.07,1699.00,22385.00,17393.00,236438.00,3.45
savings_amount_median,13843,0,446.21,519.46,0.00,358.00,734.00,9333.00,2.40
daysonmarket_median,13843,0,49.15,56.06,0.00,36.00,25.00,1987.00,8.75
mileage_median,13842,1,51463.27,51845.46,0.00,33596.50,96890.75,238541.00,0.68
year_median,13843,0,2015.95,4.72,1960.00,2017.00,7.00,2021.00,-2.25
vehicle_age_median,13842,1,4.08,4.63,0.00,3.00,7.00,60.00,2.32
owner_count_median,13804,39,1.43,0.62,1.00,1.00,1.00,6.00,1.30
horsepower_median,13837,6,240.90,56.99,78.00,243.00,95.00,630.00,0.85
engine_displacement_median,10450,3393,2925.08,808.08,1200.00,2500.00,1100.00,6800.00,1.29
